# ML-08 - Capstone Modeling Lane

Lane: **Refresh / Content Opportunity Scoring**.

The Week-4 baseline is a transparent queue rule. This notebook trains a first learned ranking model and compares it against that baseline on the same client-held-out test split and the same top-K metrics.

## 1. Method choice and why

My lane asks **which pages should a reviewer inspect first**, so I treat this as a ranking problem. The starter proxy label is `is_declining_label = trend_direction == "down"`; the model outputs a probability score, and I evaluate whether the top of the ranked list contains declining pages.

I use **Logistic Regression** as the main model because it is readable, stable with a fixed seed, and gives coefficients I can inspect. I also train a small **Random Forest** as a complexity check. If the Random Forest does not beat Logistic Regression on the same split and top-K metrics, I will not reward it just for being fancier.

Leakage guard: `trend_direction`, `trend_pct`, and the last/previous-30-day trend window columns are target-derived for this starter label, so they are excluded from features.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

ROOT = Path.cwd()
if not (ROOT / "data/raw/content_refresh_anonymized.csv").exists():
    ROOT = Path.cwd().parents[1]

DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"
OUTPUT_DIR = ROOT / "work/outputs"
METRICS_PATH = OUTPUT_DIR / "w05_model_metrics.json"
PREDICTIONS_PATH = OUTPUT_DIR / "w05_model_test_predictions.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print(f"rows n={len(df):,}")
print(f"clients n={df['client_id'].nunique():,}")
print(f"declining proxy base rate={df['is_declining_label'].mean():.3f}")

rows n=30,000
clients n=32
declining proxy base rate=0.542


## 2. Split design

I use a **client-grouped holdout split**: 75% of rows for training and 25% for testing, grouped by `client_id`. This is more honest than a random row split because pages from the same client can share content strategy, measurement behavior, and traffic patterns. Testing on held-out clients asks whether the ranking logic generalizes beyond the clients it saw during training.

The baseline rule is not trained, but it is evaluated only on the same held-out rows as the models.

In [2]:
for raw_col, log_col in [
    ("impressions_90d", "log_impressions_90d"),
    ("clicks_90d", "log_clicks_90d"),
    ("sessions_90d", "log_sessions_90d"),
    ("ai_sessions_90d", "log_ai_sessions_90d"),
]:
    df[log_col] = np.log1p(df[raw_col])

df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_position_data"] = (df["avg_position"] > 0).astype(int)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "has_word_count",
    "has_keyword_data",
    "has_position_data",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

forbidden_features = {
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_id",
    "client_id",
}
used_features = set(numeric_features + categorical_features)
leaked = sorted(used_features & forbidden_features)
assert not leaked, leaked

y = df["is_declining_label"].astype(int)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(df, y, groups=df["client_id"]))

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(f"train rows n={len(train):,}; test rows n={len(test):,}")
print(f"train clients n={train['client_id'].nunique():,}; test clients n={test['client_id'].nunique():,}")
print(f"train base rate={y_train.mean():.3f}; test base rate={y_test.mean():.3f}")
print(f"forbidden features used={leaked}")

train rows n=22,885; test rows n=7,115
train clients n=24; test clients n=8
train base rate=0.550; test base rate=0.517
forbidden features used=[]


## 3. Train + compare vs my baseline

The baseline score below is the same Week-4 rule: visible pages in reachable positions get priority when they also have low CTR, stale update recency, and a higher-risk age tier. I recompute it here so the model and baseline are evaluated on the exact same held-out rows.

Main metric: **precision@50**, with precision@10, precision@20, precision@100, average precision, and ROC AUC as supporting checks.

In [3]:
def baseline_action_score(frame):
    visible_band = ((frame["impressions_90d"] >= 300) & (frame["impressions_90d"] < 30000)).astype(int)
    reachable_position = ((frame["avg_position"] > 3) & (frame["avg_position"] <= 20)).astype(int)
    low_ctr_flag = (frame["ctr"] < 0.5).astype(int)
    stale_flag = (frame["days_since_last_update"] >= 91).astype(int)
    fresh_age_flag = frame["age_tier"].isin(["31-90", "91-180"]).astype(int)
    return (
        visible_band
        * reachable_position
        * np.log1p(frame["impressions_90d"])
        * (1 + 0.45 * stale_flag + 0.35 * low_ctr_flag + 0.25 * fresh_age_flag)
    )


def precision_at_k(y_true, scores, k):
    ranked = (
        pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
        .sort_values("score", ascending=False)
        .head(k)
    )
    return float(ranked["y"].mean()) if len(ranked) else 0.0


def ranking_metrics(name, y_true, scores):
    payload = {
        "method": name,
        "test_base_rate": float(np.mean(y_true)),
        "precision_at_10": precision_at_k(y_true, scores, 10),
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "precision_at_100": precision_at_k(y_true, scores, 100),
        "average_precision": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
    }
    return payload


preprocess = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_features,
        ),
    ]
)

models = {
    "week4_baseline_rule": None,
    "logistic_regression": Pipeline(
        [
            ("preprocess", preprocess),
            (
                "model",
                LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
            ),
        ]
    ),
    "random_forest_check": Pipeline(
        [
            ("preprocess", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=8,
                    min_samples_leaf=30,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

X_train = train[numeric_features + categorical_features]
X_test = test[numeric_features + categorical_features]

scores = {
    "week4_baseline_rule": np.asarray(baseline_action_score(test)),
}
fitted_models = {}

for model_name, model in models.items():
    if model is None:
        continue
    model.fit(X_train, y_train)
    fitted_models[model_name] = model
    scores[model_name] = model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame(
    [ranking_metrics(name, y_test, score_values) for name, score_values in scores.items()]
).sort_values(["precision_at_50", "average_precision", "roc_auc"], ascending=False)

print(comparison.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

model_of_record = "logistic_regression"
print(f"Model of record: {model_of_record}")

             method  test_base_rate  precision_at_10  precision_at_20  precision_at_50  precision_at_100  average_precision  roc_auc
logistic_regression           0.517            0.900            0.750            0.760             0.700              0.607    0.612
random_forest_check           0.517            0.500            0.400            0.540             0.570              0.590    0.605
week4_baseline_rule           0.517            0.800            0.650            0.420             0.540              0.519    0.514
Model of record: logistic_regression


## 4. Errors and interpretation

I inspect the model of record, Logistic Regression. I look at:

- the largest absolute coefficients, as a readable importance check;
- false positives near the top of the model queue;
- false negatives that the model ranked low;
- error rate by broad content and position groups.

In [4]:
logistic_model = fitted_models[model_of_record]
feature_names = logistic_model.named_steps["preprocess"].get_feature_names_out()
coefs = logistic_model.named_steps["model"].coef_[0]

importance = (
    pd.DataFrame({"feature": feature_names, "coefficient": coefs, "abs_coefficient": np.abs(coefs)})
    .sort_values("abs_coefficient", ascending=False)
    .head(12)
)
importance["feature"] = importance["feature"].str.replace("num__", "", regex=False).str.replace("cat__", "", regex=False)

test_review = test[
    [
        "content_id",
        "content_type",
        "impression_tier",
        "position_tier",
        "age_tier",
        "freshness_tier",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "is_declining_label",
        "trend_direction",
    ]
].copy()
test_review["model_score"] = scores[model_of_record]
test_review["baseline_score"] = scores["week4_baseline_rule"]
test_review["model_rank"] = test_review["model_score"].rank(method="first", ascending=False).astype(int)
test_review["pred_at_050"] = (test_review["model_score"] >= 0.50).astype(int)
test_review["error_type"] = np.select(
    [
        test_review["pred_at_050"].eq(1) & test_review["is_declining_label"].eq(0),
        test_review["pred_at_050"].eq(0) & test_review["is_declining_label"].eq(1),
    ],
    ["false_positive", "false_negative"],
    default="correct_at_050",
)

top_false_positives = (
    test_review[test_review["is_declining_label"].eq(0)]
    .sort_values("model_score", ascending=False)
    .head(3)
    .copy()
)
top_false_positives["why_hard"] = "looks visible/reachable enough for review, but starter proxy is not declining"

missed_declines = (
    test_review[test_review["is_declining_label"].eq(1)]
    .sort_values("model_score", ascending=True)
    .head(3)
    .copy()
)
missed_declines["why_hard"] = "declining by proxy, but current observable signals look weak or outside the rule/model comfort zone"

error_by_group = (
    test_review.groupby(["content_type", "position_tier"], dropna=False)
    .agg(
        n=("content_id", "size"),
        decline_rate=("is_declining_label", "mean"),
        error_rate=("error_type", lambda s: float((s != "correct_at_050").mean())),
        median_model_score=("model_score", "median"),
    )
    .reset_index()
    .query("n >= 50")
    .sort_values("error_rate", ascending=False)
    .head(10)
)

print("Top coefficient checks:")
print(importance[["feature", "coefficient"]].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nHigh-ranked false positives:")
print(
    top_false_positives[
        [
            "content_id",
            "model_rank",
            "model_score",
            "trend_direction",
            "impressions_90d",
            "ctr",
            "avg_position",
            "freshness_tier",
            "why_hard",
        ]
    ].to_string(index=False, float_format=lambda x: f"{x:.3f}")
)

print("\nLow-ranked false negatives:")
print(
    missed_declines[
        [
            "content_id",
            "model_rank",
            "model_score",
            "trend_direction",
            "impressions_90d",
            "ctr",
            "avg_position",
            "freshness_tier",
            "why_hard",
        ]
    ].to_string(index=False, float_format=lambda x: f"{x:.3f}")
)

print("\nLargest error-rate groups with n >= 50:")
print(error_by_group.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

test_predictions_out = test_review.sort_values("model_rank").copy()
test_predictions_out.to_csv(PREDICTIONS_PATH, index=False)

metrics_payload = {
    "random_state": RANDOM_STATE,
    "split": {
        "strategy": "GroupShuffleSplit by client_id",
        "train_rows": int(len(train)),
        "test_rows": int(len(test)),
        "train_clients": int(train["client_id"].nunique()),
        "test_clients": int(test["client_id"].nunique()),
        "test_base_rate": float(y_test.mean()),
    },
    "model_of_record": model_of_record,
    "comparison": comparison.to_dict(orient="records"),
    "top_coefficients": importance[["feature", "coefficient"]].to_dict(orient="records"),
    "forbidden_features_used": leaked,
    "predictions_path": "work/outputs/w05_model_test_predictions.csv",
}
METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

print(f"\nWrote work/outputs/w05_model_test_predictions.csv")
print(f"Wrote work/outputs/w05_model_metrics.json")

Top coefficient checks:
                  feature  coefficient
      log_impressions_90d        1.239
        has_position_data        0.919
      impression_tier_low        0.690
word_count_tier_1000-2000        0.658
           log_clicks_90d       -0.554
     freshness_tier_31-90       -0.552
impression_tier_excellent       -0.468
             avg_position       -0.444
         content_age_days       -0.371
               word_count        0.352
word_count_tier_2000-3500       -0.305
      freshness_tier_0-30        0.251

High-ranked false positives:
          content_id  model_rank  model_score trend_direction  impressions_90d   ctr  avg_position freshness_tier                                                                      why_hard
content_7be5f150dc65           3        0.953              up              290 0.000         5.900           0-30 looks visible/reachable enough for review, but starter proxy is not declining
content_619acf4bbcc3          11        0.931          

## 5. Self-check

- tick - Every section above is filled with markdown thinking and code that backs it
- tick - The notebook runs top to bottom with no errors
- tick - Baseline and model use the same held-out split and same metrics
- tick - The split is grouped by `client_id`
- tick - No label-derived or future-window inputs are used as features
- tick - Errors and feature interpretation are reviewed
- tick - No client names, URLs, private queries, or datasets are committed